# The price, and the error that scores it

The sale is in Ames, Iowa. For each house the question is the one the contest asks: predict `SalePrice`, the dollar price of the sale.

The training file holds 1460 sales whose prices are known. The test file holds 1459 houses and leaves the price out. A sample file shows the two columns an answer file uses, `Id` and `SalePrice`. That sample is a benchmark someone already computed. It is not a score, because the test prices are not in the file.

The contest scores a prediction with the root mean squared error of the logarithms. For positive prices $y_i$ and predictions $\hat y_i$,

$$
\mathrm{RMSLE}
= \sqrt{\dfrac{1}{n}\sum_{i=1}^{n}\big(\ln \hat y_i - \ln y_i\big)^2}.
$$

Squaring a log gap treats a cheap house and an expensive house on the same multiplicative scale. A prediction that is twice the price and a prediction that is half the price have the same squared log gap, because $\ln 2 = -\ln \tfrac{1}{2}$ after squaring.

Try it on three toy prices, $100$, $1000$, and $10000$, all predicted as $1000$. That prediction is their geometric mean:

$$
(100 \cdot 1000 \cdot 10000)^{1/3} = (10^{9})^{1/3} = 10^{3} = 1000.
$$

The log gaps are $-\ln 10$, $0$, and $\ln 10$. The mean of the squared gaps is

$$
\dfrac{2(\ln 10)^2}{3},
$$

so

$$
\mathrm{RMSLE} = \ln 10 \cdot \sqrt{\dfrac{2}{3}}.
$$

Their arithmetic mean is $(100+1000+10000)/3 = 3700$. That constant sits farther from the two ends, so its RMSLE is larger. The log error prefers the geometric mean.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# The three files, then the toy log error.
import math
print(len(train), len(test), "SalePrice" in test[0])
assert len(train) == 1460
assert len(test) == 1459
assert len(submission) == 1459
assert "SalePrice" not in test[0]
assert list(submission[0]) == ["Id", "SalePrice"]

prices = [100, 1000, 10000]
geometric = 1000
arithmetic = (100 + 1000 + 10000) / 3
assert arithmetic == 3700

def rmsle(prediction, actual):
    total = 0
    for pred, price in zip(prediction, actual):
        gap = math.log(pred) - math.log(price)
        total += gap * gap
    return math.sqrt(total / len(actual))

geo_error = rmsle([geometric] * 3, prices)
mean_error = rmsle([arithmetic] * 3, prices)
exact = math.log(10) * math.sqrt(2 / 3)
print(geo_error, mean_error, exact)
assert abs(geo_error - exact) < 1e-12
assert mean_error > geo_error


1460 1459 False
1.8800528557247886 2.2904876134335934 1.8800528557247886


## Pitfall

$\ln 10 \cdot \sqrt{2/3}$ is the error of one constant on three toy prices. It is not a contest score. The 1459 test prices are absent, so nothing in this path is a leaderboard entry.

## Summary

Predict `SalePrice`. The contest squares the gap between $\ln \hat y$ and $\ln y$. On prices $100$, $1000$, and $10000$, the geometric mean $1000$ has RMSLE $\ln 10 \cdot \sqrt{2/3}$, and the arithmetic mean $3700$ is worse.
